# 08 Spatial Epidemiology — Reference Solutions

Complete solutions for the spatial analysis exercises on the Legionnaires' disease cluster at Songbai Nursing Home.

In [ ]:
# Google Colab setup -- skip this cell if running locally
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
import pathlib

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# -- CJK font setup (prevents Chinese labels from showing as boxes) --
# Scan system font directories and explicitly register CJK fonts (more reliable than the cache)
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 150

df = pd.read_csv("data/synthetic/legionella_outbreak.csv")
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)
df["died"] = (df["outcome"] == "dead").astype(int)

## Question 1: Spatial Distribution of Case Fatality Rates

In [ ]:
# Compute floor × wing case fatality rates
spatial = df.groupby(["floor", "wing"]).agg(
    total=("case_id", "count"),
    infected=("infected", "sum"),
    died=("died", "sum"),
).reset_index()
spatial["attack_rate"] = (spatial["infected"] / spatial["total"] * 100).round(1)
spatial["cfr"] = (spatial["died"] / spatial["infected"] * 100).round(1)

print("=== Wing attack rates & case fatality rates ===")
print(spatial[["floor", "wing", "total", "infected", "died", "attack_rate", "cfr"]].to_string(index=False))

# CFR heatmap
heatmap_cfr = spatial.pivot(index="floor", columns="wing", values="cfr")

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(heatmap_cfr, annot=True, fmt=".1f", cmap="Reds",
            cbar_kws={"label": "%"}, ax=ax)
ax.set_title("Case Fatality Rate (%) by Floor × Wing")
ax.set_ylabel("Floor")
plt.tight_layout()
plt.show()

# Interpretation
highest_cfr = spatial.loc[spatial["cfr"].idxmax()]
highest_ar = spatial.loc[spatial["attack_rate"].idxmax()]
print(f"\nHighest CFR: {highest_cfr['floor']}F-{highest_cfr['wing']} ({highest_cfr['cfr']}%)")
print(f"Highest attack rate: {highest_ar['floor']}F-{highest_ar['wing']} ({highest_ar['attack_rate']}%)")
print("\n→ The wing with the highest CFR isn't necessarily the one with the highest attack rate")
print("→ CFR is also affected by resident characteristics (age, comorbidities), not just exposure intensity")

## Question 2: Spatial Distribution of Shower Use

In [ ]:
# Shower-use proportion
shower = df.groupby(["floor", "wing"]).agg(
    total=("case_id", "count"),
    shower_users=("shower_use", "sum"),
    infected=("infected", "sum"),
).reset_index()
shower["shower_pct"] = (shower["shower_users"] / shower["total"] * 100).round(1)
shower["attack_rate"] = (shower["infected"] / shower["total"] * 100).round(1)

print("=== Shower proportion vs. attack rate ===")
print(shower[["floor", "wing", "shower_pct", "attack_rate"]].to_string(index=False))

# Side-by-side heatmaps
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

hm_shower = shower.pivot(index="floor", columns="wing", values="shower_pct")
sns.heatmap(hm_shower, annot=True, fmt=".1f", cmap="Blues",
            cbar_kws={"label": "%"}, ax=axes[0])
axes[0].set_title("Shower-Use Proportion (%)")
axes[0].set_ylabel("Floor")

hm_ar = shower.pivot(index="floor", columns="wing", values="attack_rate")
sns.heatmap(hm_ar, annot=True, fmt=".1f", cmap="YlOrRd",
            cbar_kws={"label": "%"}, ax=axes[1])
axes[1].set_title("Attack Rate (%)")
axes[1].set_ylabel("Floor")

plt.tight_layout()
plt.show()

# Correlation
corr = shower[["shower_pct", "attack_rate"]].corr().iloc[0, 1]
print(f"\nShower proportion vs. attack rate correlation: r = {corr:.3f}")
print("\n→ If the high-low patterns of the two heatmaps look similar, it supports the water-transmission hypothesis")
print("→ But also consider confounders (e.g. functional_status affects shower ability, analyzed in Ch05)")

## Question 3 (Challenge): High-Risk Room List

In [ ]:
# Attack rate for each room
room_stats = df.groupby("room").agg(
    total=("case_id", "count"),
    infected=("infected", "sum"),
).reset_index()
room_stats["attack_rate"] = (room_stats["infected"] / room_stats["total"] * 100).round(1)

# Parse floor and wing
room_stats["floor"] = room_stats["room"].str[0].astype(int)
room_stats["wing"] = room_stats["room"].str[1]

# Filter for >= 75%
high_risk = (
    room_stats[room_stats["attack_rate"] >= 75]
    .sort_values("attack_rate", ascending=False)
    [["room", "total", "infected", "attack_rate", "floor", "wing"]]
)

print(f"=== High-risk room list (attack rate ≥ 75%) ===")
print(f"{len(high_risk)} rooms in total\n")
print(high_risk.to_string(index=False))

# Statistics by wing
print("\n=== Wing distribution of high-risk rooms ===")
wing_counts = high_risk.groupby(["floor", "wing"]).size().reset_index(name="high_risk_rooms")
print(wing_counts.to_string(index=False))

print("\n→ Submit this list to the infection control team to prioritize environmental sampling of these rooms")
print("→ Pay special attention to the wings where high-risk rooms cluster; inspect the showerheads and hot-water piping")

### Interpretation

- **CFR vs. attack rate**: the two aren't necessarily positively correlated. The attack rate reflects exposure risk; the CFR reflects host vulnerability
- **Shower × space**: if the wings with high shower usage are also the ones with high attack rates, the spatial analysis strengthens the water-transmission hypothesis
- **High-risk rooms**: high-risk rooms concentrated in a particular wing suggest that wing's water supply may be the transmission route
- **Recommended action**: culture for Legionella and conduct environmental sampling of the showerheads and hot-water piping in the high-risk wing

## Question 4 Solution

In [ ]:
# Dengue: case counts by district (Annan District, with more standing water, has higher risk)
rng = np.random.default_rng(841)
districts = ["安南區", "三民區", "北屯區", "板橋區", "中西區"]
pop = {"安南區": 190000, "三民區": 340000, "北屯區": 280000, "板橋區": 550000, "中西區": 78000}
rate_per_100k = {"安南區": 22, "三民區": 8, "北屯區": 5, "板橋區": 4, "中西區": 9}
_recs = []
for d in districts:
    n = rng.poisson(rate_per_100k[d] * pop[d] / 100000)
    for _ in range(n):
        _recs.append({"district": d, "age": int(rng.integers(5, 85)),
                      "serotype": rng.choice(["DENV-1", "DENV-2", "DENV-3"])})
dengue = pd.DataFrame(_recs)
region_pop = pd.DataFrame({"district": districts, "population": [pop[d] for d in districts]})
print(f"Dengue notifications: {len(dengue)} cases across {dengue['district'].nunique()} districts")

by_dist = dengue.groupby("district").size().reset_index(name="cases")
by_dist = by_dist.merge(region_pop, on="district")
by_dist["rate_per_100k"] = (by_dist["cases"] / by_dist["population"] * 100000).round(1)
by_dist = by_dist.sort_values("rate_per_100k", ascending=False)
print(by_dist.to_string(index=False))

top_rate = by_dist.iloc[0]
top_cases = by_dist.sort_values("cases", ascending=False).iloc[0]

fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(data=by_dist, x="district", y="rate_per_100k", color="#D97757", ax=ax)
ax.set_title("Dengue Incidence Rate by District (per 100,000)")
ax.set_ylabel("Incidence rate / 100k")
plt.tight_layout(); plt.show()

print(f"\nHighest incidence rate: {top_rate['district']} ({top_rate['rate_per_100k']}/100k, {top_rate['cases']} cases)")
print(f"Most cases: {top_cases['district']} ({top_cases['cases']} cases, {top_cases['rate_per_100k']}/100k)")
print("Interpretation: a district with more cases doesn't mean higher risk just because it has a larger population; incidence rate (population-standardized) is needed to fairly compare risk across districts.")

## Question 5 Solution

In [ ]:
# COVID-19: population and cases for a 4x5 grid of regions (rows A/B in the north have higher risk)
rng = np.random.default_rng(852)
_recs = []
for r in list("ABCD"):
    for c in range(1, 6):
        popn = int(rng.integers(2000, 6000))
        base = 0.03 + (0.05 if r in ("A", "B") else 0.0) + rng.normal(0, 0.008)
        cases = rng.binomial(popn, max(0.005, base))
        _recs.append({"region": f"{r}{c}", "row": r, "col": c, "population": popn, "cases": cases})
covid = pd.DataFrame(_recs)
print(f"COVID-19: {len(covid)} regions, total population {covid['population'].sum():,}, total cases {covid['cases'].sum()}")

covid["attack_rate_pct"] = (covid["cases"] / covid["population"] * 100).round(2)
grid = covid.pivot(index="row", columns="col", values="attack_rate_pct")

fig, ax = plt.subplots(figsize=(7, 4))
sns.heatmap(grid, annot=True, fmt=".2f", cmap="Reds", ax=ax,
            cbar_kws={"label": "Attack rate (%)"})
ax.set_title("COVID-19 Attack Rate Hotspot Map by Region")
plt.tight_layout(); plt.show()

hottest = covid.sort_values("attack_rate_pct", ascending=False).iloc[0]
north = covid[covid["row"].isin(["A", "B"])]["attack_rate_pct"].mean()
south = covid[covid["row"].isin(["C", "D"])]["attack_rate_pct"].mean()
print(f"Region with the highest attack rate: {hottest['region']} ({hottest['attack_rate_pct']}%)")
print(f"North (A/B) average {north:.2f}% vs. South (C/D) average {south:.2f}% → the hotspot is concentrated in the north")

## Question 6 Solution

In [ ]:
# Enterovirus: student counts and cases by grade and classroom at an elementary school (lower grades have higher risk)
rng = np.random.default_rng(863)
_recs = []
for g in range(1, 7):
    for cl in range(1, 6):
        students = int(rng.integers(25, 35))
        risk = max(0.02, 0.28 - g * 0.03)
        cases = rng.binomial(students, risk)
        _recs.append({"grade": g, "classroom": cl, "students": students, "cases": cases})
ev = pd.DataFrame(_recs)
print(f"Enterovirus: {ev['grade'].nunique()} grades × {ev['classroom'].nunique()} classrooms, {ev['cases'].sum()} cases total")

ev["attack_rate_pct"] = (ev["cases"] / ev["students"] * 100).round(1)
mat = ev.pivot(index="grade", columns="classroom", values="attack_rate_pct")

fig, ax = plt.subplots(figsize=(7, 4.5))
sns.heatmap(mat, annot=True, fmt=".1f", cmap="Reds", ax=ax,
            cbar_kws={"label": "Attack rate (%)"})
ax.set_title("Enterovirus Attack Rate Hotspot Map: Grade × Classroom")
ax.set_xlabel("Classroom"); ax.set_ylabel("Grade")
plt.tight_layout(); plt.show()

by_grade = ev.groupby("grade").apply(
    lambda g: 100 * g["cases"].sum() / g["students"].sum(), include_groups=False).round(1)
print("Overall attack rate by grade (%):")
print(by_grade.to_string())
print(f"\nHighest: grade {by_grade.idxmax()} ({by_grade.max()}%); lowest: grade {by_grade.idxmin()} ({by_grade.min()}%)")
print("Interpretation: lower grades have a higher attack rate, consistent with enterovirus being more common in young children and reflecting the hygiene habits and contact patterns typical of lower grades.")

## Question 7 Solution

In [ ]:
# Norovirus: seating floor plan for a 25-table banquet (tables near the seafood station have higher attack rates)
rng = np.random.default_rng(874)
_recs = []
for t in range(1, 26):
    x, y = (t - 1) % 5, (t - 1) // 5
    attendees = int(rng.integers(8, 12))
    near_seafood = (x <= 1 and y <= 1)   # bottom-left corner, near the seafood station
    ar = 0.6 if near_seafood else 0.1
    cases = rng.binomial(attendees, ar)
    _recs.append({"table": t, "x": x, "y": y, "attendees": attendees, "cases": cases})
noro = pd.DataFrame(_recs)
print(f"Norovirus banquet: {len(noro)} tables, {noro['attendees'].sum()} attendees, {noro['cases'].sum()} became ill")

noro["attack_rate"] = (noro["cases"] / noro["attendees"]).round(2)

fig, ax = plt.subplots(figsize=(6, 5))
sc = ax.scatter(noro["x"], noro["y"], s=noro["attendees"] * 25,
                c=noro["attack_rate"], cmap="Reds", edgecolor="#555", linewidth=0.6)
for _, r in noro.iterrows():
    ax.annotate(str(r["table"]), (r["x"], r["y"]), ha="center", va="center", fontsize=7)
ax.set_title("Norovirus Banquet Table Spot Map (point size = attendees, color = attack rate)")
ax.set_xlabel("Table X"); ax.set_ylabel("Table Y"); ax.invert_yaxis()
plt.colorbar(sc, ax=ax, label="Attack rate"); plt.tight_layout(); plt.show()

cluster = noro[noro["attack_rate"] >= 0.4].sort_values("attack_rate", ascending=False)
print("Tables in the high-attack-rate cluster:")
print(cluster[["table", "x", "y", "attendees", "cases", "attack_rate"]].to_string(index=False))
print("Interpretation: the high-attack-rate tables are concentrated in the bottom-left corner of the floor plan, pointing to the nearby seafood/cold-dish station as the likely contamination source.")

## Question 8 Solution

In [ ]:
# Tuberculosis: population, crowding index, and cases for 12 townships (large population differences → unstable small-denominator rates)
rng = np.random.default_rng(885)
_recs = []
for i in range(1, 13):
    popn = int(rng.integers(3000, 120000))
    crowding = round(float(rng.uniform(0.5, 2.0)), 2)
    cases = rng.poisson(15 * crowding * popn / 100000)
    _recs.append({"township": f"T{i:02d}", "population": popn,
                  "crowding_index": crowding, "cases": cases})
tb = pd.DataFrame(_recs)
print(f"Tuberculosis: {len(tb)} townships, population {tb['population'].min():,}–{tb['population'].max():,}")

tb["rate_per_100k"] = (tb["cases"] / tb["population"] * 100000).round(1)
by_cases = tb.sort_values("cases", ascending=False)["township"].head(3).tolist()
by_rate = tb.sort_values("rate_per_100k", ascending=False)["township"].head(3).tolist()
print(tb.sort_values("rate_per_100k", ascending=False).to_string(index=False))

small = tb.sort_values("population").head(3)
corr = tb["crowding_index"].corr(tb["rate_per_100k"])
print(f"\nTop 3 by case count: {by_cases}")
print(f"Top 3 by incidence rate: {by_rate}  → the two lists are {'the same' if by_cases == by_rate else 'different'}")
print(f"Townships with the smallest population: {small['township'].tolist()} (small denominator: 1–2 cases can swing the rate sharply → unstable rate)")
print(f"Crowding index vs. incidence rate correlation r = {corr:.2f} → crowding is {'positively correlated' if corr > 0.3 else 'not clearly associated'} with tuberculosis incidence")
print("Interpretation: township maps should show both incidence rate (population-standardized) and case counts, and rates for low-population areas should be annotated with confidence intervals or aggregated, to avoid being misled by extreme rates.")